# MedMNIST BreastMNIST Anomaly Detection with AdaDetect

## Import Required Libraries

In [3]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data.dataset import random_split
from torch.utils.data import Dataset, Subset, ConcatDataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib
from mpl_toolkits.axes_grid1 import ImageGrid
import pandas as pd

# Import MedMNIST
from medmnist import BreastMNIST

try:
    from .algo import EmpBH, adaptiveEmpBH
except ImportError:
    try:
        from algo import EmpBH, adaptiveEmpBH
    except ImportError:
        print("Warning: Could not import AdaDetect algorithms. Please ensure algo.py is in path.")

In [4]:
def get_fdp(ytrue, rejection_set):
    """
    Compute False Discovery Proportion (FDP) and True Discovery Proportion (TDP)
    """
    if rejection_set.size:
        fdp = np.sum(ytrue[rejection_set] == 0) / len(rejection_set)
        tdp = np.sum(ytrue[rejection_set] == 1) / np.sum(ytrue==1)
    else: 
        fdp = 0
        tdp = 0
    return fdp, tdp

## Classification Model

In [5]:
class BinaryConvNet(nn.Module):
    """
    Binary classification ConvNet adapted for MedMNIST images
    """
    def __init__(self, in_channels, img_size=28):
        super(BinaryConvNet, self).__init__()
        self.dim_target = 1
        self.img_size = img_size

        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3),
            nn.BatchNorm2d(16),
            nn.ReLU())

        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))

        self.layer3 = nn.Sequential(
            nn.Conv2d(16, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU())
        
        self.layer4 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU())

        self.layer5 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))

        # Calculate flattened size for fully connected layer
        # This depends on img_size, so we compute it
        fc_input_size = self._compute_fc_input_size(in_channels)
        
        self.fc = nn.Sequential(
            nn.Linear(fc_input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1))

    def _compute_fc_input_size(self, in_channels):
        """Compute the size of the flattened layer after convolutions"""
        # Create a dummy input to compute the size
        dummy_input = torch.zeros(1, in_channels, self.img_size, self.img_size)
        x = self.layer1(dummy_input)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        return int(np.prod(x.shape[1:]))

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [6]:
class NNClassifier(object):
    """
    Wrapper around NN model with fit() and predict_proba() functions
    """
    def __init__(self, model, batch_size=128, n_epochs=10):
        self.model = model
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.loss_fn = nn.BCEWithLogitsLoss() 
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)

        self.n_iter_no_change = 10
        self.tol = 1e-4

    def fit(self, train_dataset, val_dataset=None):
        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        
        if val_dataset is not None: 
            val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False)

        # For early stopping
        best_val_acc = 0
        it_no_change = 0

        for epoch in range(self.n_epochs):
            for x_batch, y_batch in train_loader:
                self.model.train()
                y_pred = self.model(x_batch)
                y_batch = y_batch.float().unsqueeze(-1)
                loss = self.loss_fn(y_pred, y_batch)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()

            # Early stopping: if validation accuracy has not improved by tol in n_iter_no_change steps, then stop
            if val_dataset is not None: 
                with torch.no_grad():
                    val_acc = 0
                    for x_val, y_val in val_loader:
                        self.model.eval()
                        yhat = self.model(x_val)
                        yhat = yhat.softmax(dim=-1)
                        y_val = y_val.squeeze().long()
                        acc = torch.sum(torch.argmax(yhat, dim=1) == y_val).item()
                        val_acc += acc
                    val_acc /= len(val_loader.dataset)
                
                    if val_acc > best_val_acc + self.tol:
                        best_val_acc = val_acc
                        it_no_change = 0
                    else:
                        it_no_change += 1
                        if it_no_change > self.n_iter_no_change:
                            print(f"Early stopping at epoch {epoch}")
                            break

    def predict_proba(self, test_dataset):
        test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

        with torch.no_grad():
            for x, y in test_loader: 
                self.model.eval()
                yhat_unnormed = self.model(x)
                yhat_logit = nn.functional.sigmoid(yhat_unnormed)
                return yhat_logit.numpy()

## Data Pre-processing

In [7]:
class custom_subset(Dataset):
    r"""
    Subset of a dataset at specified indices.

    Arguments:
        dataset (Dataset): The whole Dataset
        indices (sequence): Indices in the whole set selected for subset
        labels(sequence) : targets as required for the indices. will be the same length as indices
    """
    def __init__(self, dataset, indices, labels=None):
        self.dataset = torch.utils.data.Subset(dataset, indices)
        # Get targets - handle both numpy arrays and torch tensors
        targets = dataset._labels if hasattr(dataset, '_labels') else (
            dataset.labels if hasattr(dataset, 'labels') else None
        )
        if targets is None:
            # Fallback: create default targets
            self.targets = torch.ones(len(indices)) if labels is None else labels
        else:
            if isinstance(targets, np.ndarray):
                targets = torch.from_numpy(targets)
            self.targets = targets[indices] if labels is None else labels
            
    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        target = self.targets[idx]
        return (img, target)

    def __len__(self):
        return len(self.targets)

### Load BreastMNIST Dataset

In [ ]:
# Load BreastMNIST dataset
# Note: You may need to install medmnist: pip install medmnist
train_data = BreastMNIST(
    split='train',
    transform=transforms.ToTensor(),
    download=True
)

print(f"Dataset: {train_data}")
print(f"Number of samples: {len(train_data)}")
print(f"Image shape: {train_data[0][0].shape}")
print(f"Number of classes: {len(np.unique(train_data.labels))}")
print(f"Classes: {np.unique(train_data.labels)}")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    idx = i
    img, label = train_data[idx]
    ax = axes[i // 5, i % 5]
    # Handle different image formats
    if img.shape[0] == 3:  # RGB
        ax.imshow(img.permute(1, 2, 0).numpy())
    else:  # Grayscale
        ax.imshow(img.squeeze(0).numpy(), cmap='gray')
    ax.set_title(f"Label: {label.item() if hasattr(label, 'item') else label}")
    ax.axis('off')
plt.tight_layout()
plt.show()

### Select Inliers and Outliers

BreastMNIST is a binary classification dataset (benign vs malignant).
We use class 0 (benign) as inliers and class 1 (malignant) as outliers.

In [ ]:
# Convert labels to torch tensor for indexing
labels_tensor = torch.from_numpy(train_data.labels).squeeze()

# Inliers = class 0 (benign), Outliers = class 1 (malignant)
inlr_labels = [0]
outlr_labels = [1]

inlr = (labels_tensor[..., None] == torch.tensor(inlr_labels)).any(-1).nonzero(as_tuple=True)[0]
outlr = (labels_tensor[..., None] == torch.tensor(outlr_labels)).any(-1).nonzero(as_tuple=True)[0]

print(f"Number of inliers (benign): {len(inlr)}")
print(f"Number of outliers (malignant): {len(outlr)}")

## Apply AdaDetect Algorithm

Running AdaDetect over multiple runs to detect anomalies (malignant) among normal (benign) images.

In [ ]:
# AdaDetect parameters
level = 0.1  # False discovery rate level
test_size = 200
null_prop = 0.90
n_inlr = int(test_size * null_prop)
n_outlr = test_size - n_inlr
calib_size = 100
train_size = 300

n_runs = 5  # Number of runs to average results

fdp, tdp = [0.]*n_runs, [0.]*n_runs

# Ensure we have enough samples
print(f"Required inliers: {n_inlr + calib_size + train_size}, Available: {len(inlr)}")
print(f"Required outliers: {n_outlr}, Available: {len(outlr)}")

if len(inlr) < (n_inlr + calib_size + train_size):
    print("WARNING: Not enough inlier samples. Adjusting parameters...")
    train_size = len(inlr) // 3
    calib_size = len(inlr) // 3
    test_size = max(100, len(inlr) - train_size - calib_size)

print(f"\nStarting AdaDetect with {n_runs} runs...")

for i in range(n_runs):
    print(f"\n--- Run {i+1}/{n_runs} ---")
    
    # Construct test dataset
    test_idx = np.concatenate([np.random.choice(inlr.numpy(), replace=False, size=n_inlr),
                               np.random.choice(outlr.numpy(), replace=False, size=n_outlr)])
    test_dataset = custom_subset(train_data, test_idx, torch.ones(test_size))
    
    # Construct training dataset
    train_idx = np.setdiff1d(inlr.numpy(), test_idx)
    np.random.shuffle(train_idx)
    n = train_size + calib_size
    ar1 = train_idx[:calib_size]
    ar2 = train_idx[calib_size:n]
    
    calib_dataset = custom_subset(train_data, ar1, torch.ones(calib_size))
    train_dataset = custom_subset(train_data, ar2, torch.zeros(n - calib_size))
    training_dataset = ConcatDataset([train_dataset, calib_dataset, test_dataset])
    
    # Train CNN on training dataset
    n_channels = train_data[0][0].shape[0]  # Get number of channels from data
    img_size = train_data[0][0].shape[1]  # Get image size
    
    print(f"Image channels: {n_channels}, Size: {img_size}x{img_size}")
    
    model = BinaryConvNet(in_channels=n_channels, img_size=img_size)
    clf = NNClassifier(model, batch_size=32, n_epochs=5)
    
    print("Training model...")
    clf.fit(train_dataset=training_dataset, val_dataset=None)
    
    # Compute statistics/scores and apply BH
    print("Computing predictions...")
    test_statistics = clf.predict_proba(test_dataset)
    null_statistics = clf.predict_proba(calib_dataset)
    
    # Apply empirical BH procedure
    rej_set = EmpBH(null_statistics, test_statistics, level=level)
    
    # Compute FDP and TDP
    test_labels = np.concatenate([np.zeros(n_inlr), np.ones(n_outlr)])
    fdp_, tdp_ = get_fdp(test_labels, rej_set)
    
    fdp[i] = fdp_
    tdp[i] = tdp_
    print(f"FDP: {fdp_:.4f}, TDP: {tdp_:.4f}")

print("\n" + "="*50)
print(f"Results over {n_runs} runs:")
print(f"FDP: {np.mean(fdp):.4f} ± {np.std(fdp):.4f}")
print(f"TDP: {np.mean(tdp):.4f} ± {np.std(tdp):.4f}")
print("="*50)

## Visualization of Results

Visualize accepted and rejected samples with rejected samples highlighted with colored borders.

In [ ]:
# Note: Using the last test_dataset and rej_set from the final run

idx_rej = rej_set
idx_acc = np.setdiff1d(np.arange(len(test_dataset)), idx_rej)

# Sample from rejected and accepted
n_rej_sample = min(10, len(idx_rej))
n_acc_sample = min(38, len(idx_acc))

sample_rej = np.random.choice(idx_rej, replace=False, size=n_rej_sample) if len(idx_rej) > 0 else np.array([])
sample_acc = np.random.choice(idx_acc, replace=False, size=n_acc_sample) if len(idx_acc) > 0 else np.array([])

samples = np.concatenate([sample_rej, sample_acc])
np.random.shuffle(samples)

print(f"Total rejected: {len(idx_rej)}, Total accepted: {len(idx_acc)}")
print(f"Visualizing {len(sample_rej)} rejected and {len(sample_acc)} accepted samples")

# Visualize
n_cols = 6
n_rows = (len(samples) + n_cols - 1) // n_cols

fig = plt.figure(figsize=(10, 2*n_rows))
grid = ImageGrid(fig, 111,
                 nrows_ncols=(n_rows, n_cols),
                 axes_pad=0.1)

for ax, idx in zip(grid, samples):
    img, _ = test_dataset[int(idx)]
    
    # Handle different image formats
    if img.shape[0] == 3:  # RGB
        ax.imshow(img.permute(1, 2, 0).numpy())
    else:  # Grayscale
        ax.imshow(img.squeeze(0).numpy(), cmap='gray')
    
    ax.tick_params(bottom=False, top=False, left=False, right=False,
                   labelbottom=False, labeltop=False, labelleft=False, labelright=False)
    
    # Highlight rejections with red border
    if int(idx) in sample_rej:
        for spine in ax.spines.values():
            spine.set_color('crimson')
            spine.set_linewidth(4)

plt.suptitle('Medical Images - Rejected (red border) vs Accepted', fontsize=14)
plt.tight_layout()
plt.show()